#### In this file I explore the data to understand its structure
I carry out an exploratory data analysis

In [11]:
import pandas as pd
import sqlite3
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))
from config import DATA_FILE, DB_FILE

In [ ]:
#Structure of my data

conn = sqlite3.connect(DB_FILE)

df_gl = pd.read_sql("SELECT * FROM gl", conn)
df_coa = pd.read_sql("SELECT * FROM coa", conn)
df_calendar = pd.read_sql("SELECT * FROM calendar", conn)
df_territory = pd.read_sql("SELECT * FROM territory", conn)
df_cashflow = pd.read_sql("SELECT * FROM cashflow", conn)
df_soce_st = pd.read_sql("SELECT * FROM soce_st", conn)


print(df_gl.shape)        
print(df_gl.dtypes)       
print(df_gl.columns)   

print("-------")

print(df_coa.shape)        
print(df_coa.dtypes)       
print(df_coa.columns)

print("-------")

print(df_calendar.shape)        
print(df_calendar.dtypes)       
print(df_calendar.columns)

print("-------")

print(df_territory.shape)        
print(df_territory.dtypes)       
print(df_territory.columns)

print("-------")

print(df_cashflow.shape)        
print(df_cashflow.dtypes)       
print(df_cashflow.columns)

print("-------")

print(df_soce_st.shape)        
print(df_soce_st.dtypes)       
print(df_soce_st.columns)



(27909, 6)
EntryNo          float64
Date              object
Territory_key      int64
Account_key        int64
Details           object
Amount             int64
dtype: object
Index(['EntryNo', 'Date', 'Territory_key', 'Account_key', 'Details', 'Amount'], dtype='object')
-------
(54, 7)
Account_key     int64
Report         object
Class          object
SubClass       object
SubClass2      object
Account        object
SubAccount     object
dtype: object
Index(['Account_key', 'Report', 'Class', 'SubClass', 'SubClass2', 'Account',
       'SubAccount'],
      dtype='object')
-------
(1096, 5)
Date       object
Year        int64
Quarter    object
Month      object
Day        object
dtype: object
Index(['Date', 'Year', 'Quarter', 'Month', 'Day'], dtype='object')
-------
(7, 3)
Territory_key     int64
Country          object
Region           object
dtype: object
Index(['Territory_key', 'Country', 'Region'], dtype='object')
-------
(66, 7)
Type           object
Subtype        object
Rank        

In [8]:
#Quality of my data

print(df_gl.isnull().sum())
print(df_gl.duplicated().sum())
print(df_gl.describe())

print("------")

print(df_coa.isnull().sum())
print(df_coa.duplicated().sum())
print(df_coa.describe())

print("------")

print(df_calendar.isnull().sum())
print(df_calendar.duplicated().sum())
print(df_calendar.describe())

print("------")

print(df_territory.isnull().sum())
print(df_territory.duplicated().sum())
print(df_territory.describe())

print("------")

print(df_cashflow.isnull().sum())
print(df_cashflow.duplicated().sum())
print(df_cashflow.describe())

print("------")

print(df_soce_st.isnull().sum())
print(df_soce_st.duplicated().sum())
print(df_soce_st.describe())


EntryNo          0
Date             0
Territory_key    0
Account_key      0
Details          0
Amount           0
dtype: int64
0
            EntryNo  Territory_key   Account_key        Amount
count  27909.000000   27909.000000  27909.000000  2.790900e+04
mean     995.992673       4.000000    144.533735  8.828694e+02
std      574.544811       2.000036    143.986797  2.705571e+04
min        1.100000       1.000000     10.000000 -1.092000e+06
25%      499.100000       2.000000     30.000000 -2.291000e+03
50%      995.200000       4.000000    120.000000 -5.280000e+02
75%     1493.200000       6.000000    230.000000  1.651000e+03
max     1992.300000       7.000000   1010.000000  1.092000e+06
------
Account_key    0
Report         0
Class          0
SubClass       0
SubClass2      0
Account        0
SubAccount     0
dtype: int64
0
       Account_key
count    54.000000
mean    232.000000
std     170.243222
min      10.000000
25%     112.500000
50%     205.500000
75%     337.500000
max    1010

No missing data in this test dataset.

In [ ]:
#Which data is actually in my dataset?

pd.read_sql("""
    SELECT Report, Class, SubClass, COUNT(*) AS account_count
    FROM coa
    GROUP BY Report, Class, SubClass
""", conn)


,Report,Class,SubClass,account_count
0,Adjusting,Adjusting,Adjusting,1
1,Balance Sheet,Assets,Assets,13
2,Balance Sheet,Liabilities and Owners Equity,Liabilities,10
3,Balance Sheet,Liabilities and Owners Equity,Owners Equity,4
4,Profit and Loss,Interest & Tax,Interest Expense,1
5,Profit and Loss,Interest & Tax,Taxation,1
6,Profit and Loss,Non-operating,Dividend Income,1
7,Profit and Loss,Non-operating,Exchange Loss/Gain,1
8,Profit and Loss,Non-operating,Gain/Loss on Sales of Asset,1
9,Profit and Loss,Non-operating,Interest Income,1


In [14]:
#What date range does it cover?

pd.read_sql("""
    SELECT MIN(Date) AS start_date, MAX(Date) AS end_date
    FROM gl
""", conn)

,start_date,end_date
0,2018-01-01 00:00:00,2020-12-31 00:00:00


In [17]:
#What territories exist?

pd.read_sql("""
    SELECT t.Territory_key, t.Country, t.Region, COUNT(*) AS transaction_count
    FROM gl
    JOIN territory t ON gl.Territory_key = t.Territory_key
    GROUP BY t.Territory_key, t.Country, t.Region
""", conn)


,Territory_key,Country,Region,transaction_count
0,1,USA,North America,3987
1,2,Canada,North America,3987
2,3,UK,Europe,3987
3,4,Germany,Europe,3987
4,5,France,Europe,3987
5,6,Australia,Oceania,3987
6,7,New Zealand,Oceania,3987


In [18]:
# What types of transaction exist?

pd.read_sql("""
    SELECT Details, COUNT(*) AS count, SUM(Amount) AS total_amount
    FROM gl
    GROUP BY Details
    ORDER BY total_amount DESC
""", conn)


,Details,count,total_amount
0,Credit Sales,2772,28480032
1,Share Issue,42,16211734
2,Cash Sales,1764,6631830
3,New loan raised @ 6%,42,1287254
4,Inventory Purchase,994,739968
5,Purchase of intangibles assets,42,575140
6,Bad Debts,462,252282
7,Dividend income,42,135396
8,Interest income,504,121970
9,Sale of asset,63,32572


In [ ]:
#How does my P&L distribution looks like?

pd.read_sql("""
    SELECT Report, Class, SubClass, Account, SubAccount
    FROM coa
    WHERE Report = 'Profit and Loss'
    ORDER BY Class, SubClass, Account
""", conn)

,Report,Class,SubClass,Account,SubAccount
0,Profit and Loss,Interest & Tax,Interest Expense,Interest Expense,Interest Expense
1,Profit and Loss,Interest & Tax,Taxation,Taxation,Taxation
2,Profit and Loss,Non-operating,Dividend Income,Dividend Income,Dividend Income
3,Profit and Loss,Non-operating,Exchange Loss/Gain,Exchange Loss/Gain,Exchange Loss/Gain
4,Profit and Loss,Non-operating,Gain/Loss on Sales of Asset,Gain/Loss on Sales of Asset,Gain/Loss on Sales of Asset
5,Profit and Loss,Non-operating,Interest Income,Interest Income,Interest Income
6,Profit and Loss,Operating account,Depreciation & Amortization,Amortization of Intangible Assets,Amortization of Intangible Assets
7,Profit and Loss,Operating account,Depreciation & Amortization,Equipment,Equipment
8,Profit and Loss,Operating account,Depreciation & Amortization,Furniture and Fixtures,Furniture and Fixtures
9,Profit and Loss,Operating account,Depreciation & Amortization,Vehicles,Vehicles
